# Pair selection 2: same sector and strong link

**Why a second notebook.** The first selection ([coppie.ipynb](../Old%20files/src/01_selezione_coppie/coppie.ipynb), now in `Old files/`, in Italian) chose the pairs with the price distance only. Here we start again from scratch with a more targeted criterion: **same sector and strong link**, with stocks that really trade. The first notebook stays as it is, as a comparison.

**What it produces:** the list of pairs of **version 2** (up to 15). The group decided to **keep all 15**, even though the plan foresaw 6-9: the difference must be stated in the report.

**The group's criteria, and how they become measurable rules**

| Criterion | Rule in this notebook | Data used |
|---|---|---|
| Same sector, recognisable economic link | same GICS **sub-industry** (narrower than the industry group used before) | `gics.csv` |
| Strong link | correlation of returns between 0.6 and 0.95, then ranking by price **distance** (SSD) | `daily` |
| Stocks that really trade | median daily traded value ≥ 100 million $ for *both* stocks | `daily` |
| *Not filters* | **costs** (the professor's indication), **news** (another step), **structural events** (pairs that have them stay, to be flagged in the report) | |

**Anti data-snooping rule (what we do and what we do not do)**
1. **Distances, correlations and daily liquidity** use only daily data up to **15 June 2026**.
2. **The initial list is decided by the rule**, applied mechanically by the cell of part 5: ranking by distance, one stock per pair, cap per sector. No pair is added, removed or replaced by hand.
3. **The thresholds are fixed in the parameter cell** and are not retouched after looking at the results (Lo and MacKinlay, 1990; Harvey, Liu and Zhu, 2016).
4. **To be stated in the report:** before writing this notebook we looked at the data once (liquidity distribution of the stocks and a provisional list of candidate pairs), and the group later removed the filters on news, costs, events and the exclusion of stocks already used, after seeing the first runs. The thresholds are not optimised on the final list, but they are not independent of the data either.
5. **The diagnostics of part 6 are not filters**: they describe the pairs, but the list is decided by the rule only.

## 1. Parameters, with the reason for each choice

Legend: **[ECO]** economic reason, **[ACA]** academic reference, **[OURS]** our own choice without a source: it must be stated as such.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # project folder: makes src importable

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint
from statsmodels.stats.multitest import multipletests

from src.config import PROCESSED, SELECTED
from src.data import load
from src.gics import load_gics

# ---- Selection period ----
SEL_START = "2021-06-16"
SEL_END   = "2026-06-15"
# [ACA/ECO] It ends on 15 June 2026: the selection uses only daily data and must not see the following period.
# [OURS] 5 years, as in the first selection, so that they can be compared; it starts in 2021 to leave out the Covid shock.

# ---- Filters on the single stock ----
MIN_DOLLAR_VOLUME = 100e6
# [ECO] We need stocks that can be bought and sold without moving the price. Twice the threshold of the first selection
#   (50 million) because here liquidity is an explicit criterion. [OURS] the threshold: there is no official value.
# No filter on news, costs and structural events: the group's choice (costs are not to be considered, the professor's indication).
#
MAX_MISSING_PCT = 0.01
# [ECO] Complete data: without holes the distance between the pairs is not distorted.

# ---- Filters on the pair ----
MIN_CORR = 0.6
MAX_CORR = 0.95
# [ECO] Below 0.6 the link is weak; above 0.95 the two stocks are almost the same share (two share classes of the
#   same company) and there is no spread to exploit. Same thresholds as the first selection.
PAIR_LEVEL = "sub_industry"
# [ECO] Criterion 3: the GICS sub-industry (e.g. "Semiconductors") is narrower than the industry group ("Semiconductors &
#   Semiconductor Equipment") and makes the economic link more recognisable. Limit: today's GICS, not historical.
CAP_LEVEL = "industry_group"
MAX_PER_INDUSTRY = 3
# [OURS] at most 3 pairs per industry group, so as not to concentrate everything in one sector.
N_PAIRS = 15
# [OURS] 15 pairs, all kept by the group's decision (the plan foresaw 6-9: to be stated in the report).

## 2. Data

- **Daily** (up to 15 June 2026): close price adjusted for splits and dividends (`PX_LAST`) and volume, for prices, returns and traded value.

In [2]:
daily = load("daily")
daily["ticker"] = daily["ticker"].astype(str)
daily = daily[(daily["date"] >= SEL_START) & (daily["date"] <= SEL_END)]
prices  = daily.pivot(index="date", columns="ticker", values="PX_LAST")
volume  = daily.pivot(index="date", columns="ticker", values="PX_VOLUME")
returns = daily.pivot(index="date", columns="ticker", values="ret")

years = len(prices) / 252
print(f"{len(prices)} trading days ({years:.1f} years), {prices.shape[1]} tickers, from {prices.index.min().date()} to {prices.index.max().date()}")
funnel = {"starting stocks": prices.shape[1]}

1255 trading days (5.0 years), 501 tickers, from 2021-06-16 to 2026-06-15


## 3. Filters on the single stock

A pair is tradable only if *both* stocks pass the filters: daily liquidity and complete data. Each filter shrinks the universe; the funnel goes into the report with its numbers.

**What is not filtered, by the group's choice:** **costs** (the professor's indication), **news** (another step of the plan) and **structural events** (mergers, acquisitions, spin-offs): the pairs that have them stay in the list and must be flagged in the report.

In [3]:
def short(t): return t.split()[0]

dollar_volume = (prices * volume).median() / 1e6          # in millions of $
missing = prices.isna().mean()
t = pd.DataFrame({"dv_m": dollar_volume, "missing": missing})
t["short"] = [short(x) for x in t.index]

f2 = t["dv_m"] < MIN_DOLLAR_VOLUME / 1e6
f4 = t["missing"] > MAX_MISSING_PCT

keep = ~(f2 | f4)
funnel["liquid (daily traded value)"] = int((~f2).sum())
funnel["admitted stocks (few holes)"] = int(keep.sum())
print(pd.Series(funnel, name="n").to_frame())

                               n
starting stocks              501
liquid (daily traded value)  166
admitted stocks (few holes)  154


## 4. GICS sector and final universe

`data/gics.csv` (from Bloomberg) gives `industry_group` and `sub_industry`; the export is in Italian and `load_gics` ([gics.py](gics.py)) translates the names to the official English GICS names. For a pair the **same sub-industry** is required; for the cap on the number of pairs the industry group is used. **Limit to be stated:** current classification, without history.

In [4]:
g = load_gics()
univ = t[keep].index
px = prices[univ].ffill()
px = px.loc[:, px.notna().all()]                     # prices are needed from the first to the last day
rets = returns[px.columns]
names = list(px.columns)
sub = pd.Series({n: g.loc[short(n), PAIR_LEVEL] if short(n) in g.index else None for n in names})
grp = pd.Series({n: g.loc[short(n), CAP_LEVEL] if short(n) in g.index else None for n in names})
print(f"{len(names)} stocks in the final universe; {sub.nunique()} sub-industries, of which {(sub.value_counts() >= 2).sum()} with at least 2 stocks")
funnel["with full history and GICS"] = int(sub.notna().sum())

154 stocks in the final universe; 64 sub-industries, of which 38 with at least 2 stocks


## 5. Pair selection: distance method, mechanical rule

**Idea** (Gatev, Goetzmann and Rouwenhorst, 2006): if two stocks were "close" in the past, they probably stay so; when they drift apart, one bets on the return. As in the first selection: each price is divided by its first value ($1 invested at the start), the **SSD** is the sum of the squared differences between the two indices, and pairs are sorted by increasing SSD.

**What changes compared with the first time:** the candidates are only the pairs of the **same sub-industry** among the stocks that already passed the liquidity and complete-data filters. The ranking is then walked from the top: a pair is taken if neither stock is already used and if its industry group does not already have `MAX_PER_INDUSTRY` pairs. It stops at `N_PAIRS`. **Nothing else: no pair goes in or out by hand.**

In [5]:
norm = px / px.iloc[0]
x = norm.values.T
sq = (x ** 2).sum(axis=1)
SSD = sq[:, None] + sq[None, :] - 2 * x @ x.T
C = rets.corr().values

rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if pd.notna(sub[names[i]]) and sub[names[i]] == sub[names[j]]:
            rows.append(dict(A=names[i], B=names[j], sub_industry=sub[names[i]], industry_group=grp[names[i]], ssd=SSD[i, j], corr=C[i, j],
                             dv_min_m=min(t.loc[names[i], "dv_m"], t.loc[names[j], "dv_m"])))
same = pd.DataFrame(rows).sort_values("ssd").reset_index(drop=True)
same["rank_ssd"] = np.arange(1, len(same) + 1)
dist_funnel = {"pairs of the same sub-industry among the admitted stocks": len(same)}

identical = same[same["corr"] > MAX_CORR]
cand = same[(same["corr"] <= MAX_CORR) & (same["corr"] >= MIN_CORR)]
dist_funnel[f"correlation between {MIN_CORR} and {MAX_CORR}"] = len(cand)
print("Excluded as almost identical:"); display(identical[["A", "B", "corr", "ssd"]].round(3))
print("Discarded for low correlation:", int((same["corr"] < MIN_CORR).sum()))

chosen, used, per_grp = [], set(), {}
for _, r in cand.iterrows():
    if r["A"] in used or r["B"] in used or per_grp.get(r["industry_group"], 0) >= MAX_PER_INDUSTRY:
        continue
    chosen.append(r); used.update([r["A"], r["B"]]); per_grp[r["industry_group"]] = per_grp.get(r["industry_group"], 0) + 1
    if len(chosen) == N_PAIRS:
        break
final = pd.DataFrame(chosen).reset_index(drop=True)
final.insert(0, "pair", final["A"].map(short) + "-" + final["B"].map(short))
dist_funnel[f"final pairs (max {N_PAIRS})"] = len(final)
print(f"\nPairs examined: {len(same)} (same sub-industry); candidates after the correlation filter: {len(cand)}.")
final[["pair", "sub_industry", "rank_ssd", "ssd", "corr", "dv_min_m"]].round(2)

Excluded as almost identical:


,A,B,corr,ssd
1,GOOG UW Equity,GOOGL UW Equity,0.996,3.862


Discarded for low correlation: 176

Pairs examined: 249 (same sub-industry); candidates after the correlation filter: 72.


,pair,sub_industry,rank_ssd,ssd,corr,dv_min_m
0,MA-V,Transaction & Payment Processing Services,1,3.85,0.88,319.43
1,CSX-UNP,Rail Transportation,3,6.61,0.72,137.02
2,PYPL-XYZ,Transaction & Payment Processing Services,4,8.74,0.61,119.01
3,HD-LOW,Home Improvement Retail,5,10.39,0.87,169.11
4,DHR-TMO,Life Sciences Tools & Services,6,15.92,0.81,165.60
5,CRM-WDAY,Application Software,7,18.26,0.65,154.31
6,JPM-WFC,Diversified Banks,8,20.09,0.76,261.05
7,NXPI-TXN,Semiconductors,9,21.58,0.76,186.36
8,ELV-UNH,Managed Health Care,10,22.84,0.63,137.38
9,CL-PG,Household Products,17,33.17,0.72,109.75


**How to read the table.** `rank_ssd` is the rank by distance among the pairs of the same sub-industry; `dv_min_m` is the daily traded value (millions of $) of the *worse* stock of the pair: a pair is worth as much as its weakest link. If there are fewer pairs than `N_PAIRS`, it is a result, not an error: we write it in the report and we do not widen the filters to get more.

## 6. Statistical diagnostics (not filters)

As in the first selection, **no pair is discarded** here: the list is already fixed. The measures describe the pairs and flag the riskiest ones in the report; the selection is not based on them. On daily data and over 5 years they measure horizons of days and months.

- **Cointegration p-value** (Engle-Granger on log prices, in both directions, the worse one is kept): close to zero = the spread tends to go back to the mean.
- **Hedge ratio β**: regression of the log price of A on that of B; close to 1 is consistent with equal capital on the two legs.
- **Half-life** (Ornstein-Uhlenbeck, Chan 2013): days to halve a deviation.
- **Mean crossings per year**: how many trading opportunities the spread offers.
- **Stability**: distance and β in the two halves of the period.

In [6]:
logp = np.log(px)
half_a, half_b = px.index[: len(px) // 2], px.index[len(px) // 2 :]

def half_life(s):
    lam = np.polyfit(s[:-1], np.diff(s), 1)[0]
    return -np.log(2) / lam if lam < 0 else np.inf

def crossings(s):
    s = s - s.mean()
    return int((np.sign(s[1:]) != np.sign(s[:-1])).sum())

def worst_p(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return max(coint(u, v)[1], coint(v, u)[1])

def ssd_on(a, b, idx):
    d = px.loc[idx, a] / px.loc[idx, a].iloc[0] - px.loc[idx, b] / px.loc[idx, b].iloc[0]
    return float((d ** 2).sum())

def beta_on(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return np.polyfit(v, u, 1)[0]

diag = []
for _, r in final.iterrows():
    a, b = r["A"], r["B"]
    lr = np.log(px[a] / px[b]).values
    diag.append(dict(pair=r["pair"], spread_std_pct=lr.std() * 100, coint_p=worst_p(a, b), beta=beta_on(a, b), half_life_days=half_life(lr),
                     crossings_per_year=crossings(lr) / years, ssd_first_half=ssd_on(a, b, half_a), ssd_second_half=ssd_on(a, b, half_b),
                     beta_first_half=beta_on(a, b, half_a), beta_second_half=beta_on(a, b, half_b)))
diag = pd.DataFrame(diag)
diag.round(3)

,pair,spread_std_pct,coint_p,beta,half_life_days,crossings_per_year,ssd_first_half,ssd_second_half,beta_first_half,beta_second_half
0,MA-V,3.336,0.081,0.985,32.121,14.457,0.988,1.380,0.957,0.832
1,CSX-UNP,6.339,0.505,0.950,56.479,8.433,1.342,5.246,0.736,1.522
2,PYPL-XYZ,17.732,0.557,0.985,62.121,13.453,4.941,25.585,0.924,0.676
3,HD-LOW,5.152,0.144,0.975,52.404,7.630,4.818,3.785,0.872,0.740
4,DHR-TMO,7.056,0.286,0.907,61.593,10.642,6.982,4.543,0.937,0.867
5,CRM-WDAY,13.619,0.401,0.822,67.857,10.241,6.380,29.652,0.884,0.640
6,JPM-WFC,10.600,0.172,1.167,109.697,6.425,10.523,5.827,0.587,1.035
7,NXPI-TXN,11.296,0.790,0.885,66.262,10.441,4.846,42.680,1.008,0.504
8,ELV-UNH,11.288,0.114,0.645,60.335,10.843,2.315,12.200,0.928,0.598
9,CL-PG,6.059,0.065,1.139,62.264,8.835,21.404,3.346,0.303,1.092


**Multiple testing.** The distance ranks many pairs, so some are "close by chance". The number of pairs examined (part 5) must be reported in the report.

## 7. Comparison with the first selection

Which of the pairs chosen here were already in the first selection (`selected_pairs.csv`) and which pairs of the first selection do not appear. It only serves to document in the report how much the two selections overlap.

In [7]:
OLD_SELECTED = PROCESSED / "selected_pairs.csv"    # list of the first selection: it exists only if its notebook (in Old files) was run
if not OLD_SELECTED.exists():
    print(f"{OLD_SELECTED.name} is missing: the comparison with the first selection stays empty")
old = pd.read_csv(OLD_SELECTED) if OLD_SELECTED.exists() else pd.DataFrame({"pair": []})
cmp_new = final[["pair", "sub_industry", "rank_ssd", "corr"]].copy()
cmp_new["in_first_selection"] = cmp_new["pair"].isin(set(old["pair"]))
print("Pairs of the first selection not present here:", sorted(set(old["pair"]) - set(final["pair"])))
cmp_new.round(2)

Pairs of the first selection not present here: ['BAC-PNC', 'BNY-GS', 'ITW-OTIS', 'MCD-YUM', 'TFC-USB']


,pair,sub_industry,rank_ssd,corr,in_first_selection
0,MA-V,Transaction & Payment Processing Services,1,0.88,True
1,CSX-UNP,Rail Transportation,3,0.72,True
2,PYPL-XYZ,Transaction & Payment Processing Services,4,0.61,True
3,HD-LOW,Home Improvement Retail,5,0.87,True
4,DHR-TMO,Life Sciences Tools & Services,6,0.81,False
5,CRM-WDAY,Application Software,7,0.65,False
6,JPM-WFC,Diversified Banks,8,0.76,False
7,NXPI-TXN,Semiconductors,9,0.76,False
8,ELV-UNH,Managed Health Care,10,0.63,False
9,CL-PG,Household Products,17,0.72,False


## 8. Summary for the report

**Funnel of stocks and pairs** (to be reported with the numbers):

In [8]:
display(pd.Series(funnel, name="n").to_frame())
pd.Series(dist_funnel, name="n").to_frame()

,n
starting stocks,501
liquid (daily traded value),166
admitted stocks (few holes),154
with full history and GICS,154


,n
pairs of the same sub-industry among the admitted stocks,249
correlation between 0.6 and 0.95,72
final pairs (max 15),15


**Limits to be stated in the report**
1. **Survivorship bias**: the universe is today's S&P 500 members.
2. **Current GICS**, not historical.
3. **Our own thresholds** (daily liquidity 100 million $, correlation 0.6-0.95, cap of 3 pairs per group, 15 pairs): reasonable and stated before seeing the pairs, but neither optimised nor derived from a source. Some were chosen after looking at the distribution of the stocks (a first data exploration precedes this notebook).
4. **Structural events not excluded**: the list may contain pairs with mergers, acquisitions or spin-offs in the period (by the group's choice). For these pairs the link may have changed and the reversion of the spread is less reliable: to be flagged pair by pair in the report.
5. **Costs not considered** (as indicated by the professor): no filter on price or trading cost. Liquidity is a filter anyway (point 3).
6. **News**: no filter and no column in this notebook; they are evaluated in their own step of the plan.
7. **A single selection period** of 5 years: the link can change.
8. **Multiple testing**: the number of pairs examined is in part 5.

## 9. Results and chosen pairs (text for the report)

The next cell generates the text from the results of the notebook: if it is rerun with other data or parameters, numbers and table update by themselves. The economic descriptions (`ECON`) and the notes on known events (`EVENTS`) are written by hand from general knowledge and **must be checked before putting them in the report**.

In [9]:
from IPython.display import Markdown, display

# short economic description (text written by hand, to be checked)
ECON = {
    "MA-V": "the two large card payment networks: same business model, same customers, revenues tied to consumer spending.",
    "CSX-UNP": "two US freight railroads: same freight volumes, same regulation, same costs (fuel, labour).",
    "PYPL-XYZ": "digital payments for consumers and merchants (XYZ is Block): common sector, but very wide spread and low correlation.",
    "HD-LOW": "the two large US home-improvement and DIY retailers: same customers, same seasonality, same sensitivity to the housing market.",
    "DHR-TMO": "life sciences tools and services: same customers (labs, pharma, biotech) and same spending cycles.",
    "CRM-WDAY": "subscription enterprise software (CRM and human capital management): same customers and same sensitivity to IT spending.",
    "JPM-WFC": "two large US universal banks: same rate, credit and regulatory factors.",
    "NXPI-TXN": "analog and power semiconductors: same end markets (autos, industry) and same cycle.",
    "ELV-UNH": "US health insurers (managed care): same premium revenues, same regulation and same cost of care.",
    "CL-PG": "household and personal care consumer goods: same sales channels (mass retail) and same raw-material costs.",
    "GS-MS": "the two large US investment banks: same revenues (trading, advisory, wealth management).",
    "ISRG-SYK": "medical technology for surgery: same customers (hospitals) and same capital spending; different businesses (robots versus implants).",
    "KO-PEP": "Coca-Cola and Pepsi: same soft-drink market, same customers and sales channels.",
    "BAC-C": "two large US universal banks: same rate and credit factors.",
    "COP-EOG": "oil and gas production (exploration and production): revenues tied to the same commodity prices.",
}
# known structural events in the period: they are NOT filters (the group's choice), they only serve to flag them in the report. To be checked.
EVENTS = {
    "CSX-UNP": "UNP announced the merger with Norfolk Southern (2025)",
    "HD-LOW": "HD acquired SRS Distribution (2024) and LOW FBM (2025)",
    "DHR-TMO": "DHR spun off Veralto (2023)",
    "CRM-WDAY": "CRM acquired Informatica (2025)",
    "JPM-WFC": "WFC: asset cap lifted by the Fed (2025)",
    "BAC-C": "C: sale of Banamex under way",
    "COP-EOG": "COP acquired Marathon Oil (2024)",
}

def fmt_hl(x): return "undefined" if not np.isfinite(x) else f"{x:.0f}"
res = final.merge(diag, on="pair")
res["event"] = res["pair"].map(EVENTS).fillna("none known")
n_pairs = len(res)
n_old = int(cmp_new["in_first_selection"].sum())
n_p05 = int((res["coint_p"] < 0.05).sum())
n_p10 = int((res["coint_p"] < 0.10).sum())
rows = ["| Pair | Sub-industry | SSD rank (of %d) | Correlation | Spread width | Coint. p-value | Half-life (days) | Known event |" % len(same), "|---|---|---|---|---|---|---|---|"]
for _, r in res.iterrows():
    rows.append(f"| {r['pair']} | {r['sub_industry']} | {r['rank_ssd']} | {r['corr']:.2f} | {r['spread_std_pct']:.1f}% | {r['coint_p']:.3f} | {fmt_hl(r['half_life_days'])} | {r['event']} |")
desc = "\n".join(f"- **{p}**: {ECON.get(p, '(description to be written)')}" for p in res["pair"])
text = "\n".join([
    "### What we did",
    f"Out of the {funnel['starting stocks']} stocks of the index we kept the {funnel['liquid (daily traded value)']} with a median daily traded value of at least {MIN_DOLLAR_VOLUME/1e6:.0f} million $ (stocks that really trade) and the {funnel['admitted stocks (few holes)']} with almost complete data in the period {SEL_START} - {SEL_END}. "
    f"Among these we took only pairs of the **same GICS sub-industry** ({dist_funnel['pairs of the same sub-industry among the admitted stocks']} pairs), those with a return correlation between {MIN_CORR} and {MAX_CORR} ({len(cand)} pairs), and we sorted them by price **distance** (SSD, Gatev et al., 2006). "
    f"From the top we took a pair if no stock was already used and if the industry group did not already have {MAX_PER_INDUSTRY} pairs, up to {N_PAIRS}: the result is **{n_pairs}**.",
    "",
    "### Why",
    "Same sector and strong link are the economic conditions for the spread to go back to the mean: similar companies are exposed to the same factors. The distance is the reference method of the pairs-trading literature and needs no statistical model. The liquidity threshold keeps only pairs that really trade. Costs, news and structural events are not filters: the first by the professor's indication, the news because they are another step of the plan, the events by the group's choice (they are only flagged).",
    "",
    "### The chosen pairs (version 2)",
    "\n".join(rows),
    "",
    desc,
    "",
    "### What emerges",
    f"- {n_old} pairs out of {n_pairs} were already in version 1 ({', '.join(cmp_new.loc[cmp_new['in_first_selection'], 'pair'])}); the other {n_pairs - n_old} are new.",
    f"- The statistical link on daily data is weak: only {n_p05} " + ("pair has" if n_p05 == 1 else "pairs have") + f" a cointegration p-value below 0.05 ({n_p10} below 0.10). The list is a list of candidates, not a proof: the proof comes from the next steps.",
    "- Some pairs have an undefined half-life (the spread does not go back to the mean in the period) or a very different distance between the two halves of the period: see the table of part 6. They were not discarded, by rule.",
    f"- Pairs examined: {len(same)} (multiple testing: some are close by chance).",
])
display(Markdown(text))

### What we did
Out of the 501 stocks of the index we kept the 166 with a median daily traded value of at least 100 million $ (stocks that really trade) and the 154 with almost complete data in the period 2021-06-16 - 2026-06-15. Among these we took only pairs of the **same GICS sub-industry** (249 pairs), those with a return correlation between 0.6 and 0.95 (72 pairs), and we sorted them by price **distance** (SSD, Gatev et al., 2006). From the top we took a pair if no stock was already used and if the industry group did not already have 3 pairs, up to 15: the result is **15**.

### Why
Same sector and strong link are the economic conditions for the spread to go back to the mean: similar companies are exposed to the same factors. The distance is the reference method of the pairs-trading literature and needs no statistical model. The liquidity threshold keeps only pairs that really trade. Costs, news and structural events are not filters: the first by the professor's indication, the news because they are another step of the plan, the events by the group's choice (they are only flagged).

### The chosen pairs (version 2)
| Pair | Sub-industry | SSD rank (of 249) | Correlation | Spread width | Coint. p-value | Half-life (days) | Known event |
|---|---|---|---|---|---|---|---|
| MA-V | Transaction & Payment Processing Services | 1 | 0.88 | 3.3% | 0.081 | 32 | none known |
| CSX-UNP | Rail Transportation | 3 | 0.72 | 6.3% | 0.505 | 56 | UNP announced the merger with Norfolk Southern (2025) |
| PYPL-XYZ | Transaction & Payment Processing Services | 4 | 0.61 | 17.7% | 0.557 | 62 | none known |
| HD-LOW | Home Improvement Retail | 5 | 0.87 | 5.2% | 0.144 | 52 | HD acquired SRS Distribution (2024) and LOW FBM (2025) |
| DHR-TMO | Life Sciences Tools & Services | 6 | 0.81 | 7.1% | 0.286 | 62 | DHR spun off Veralto (2023) |
| CRM-WDAY | Application Software | 7 | 0.65 | 13.6% | 0.401 | 68 | CRM acquired Informatica (2025) |
| JPM-WFC | Diversified Banks | 8 | 0.76 | 10.6% | 0.172 | 110 | WFC: asset cap lifted by the Fed (2025) |
| NXPI-TXN | Semiconductors | 9 | 0.76 | 11.3% | 0.790 | 66 | none known |
| ELV-UNH | Managed Health Care | 10 | 0.63 | 11.3% | 0.114 | 60 | none known |
| CL-PG | Household Products | 17 | 0.72 | 6.1% | 0.065 | 62 | none known |
| GS-MS | Investment Banking & Brokerage | 24 | 0.84 | 10.2% | 0.048 | 157 | none known |
| ISRG-SYK | Health Care Equipment | 25 | 0.63 | 14.2% | 0.084 | 110 | none known |
| KO-PEP | Soft Drinks & Non-alcoholic Beverages | 27 | 0.68 | 15.4% | 0.946 | undefined | none known |
| BAC-C | Diversified Banks | 29 | 0.81 | 16.5% | 0.587 | undefined | C: sale of Banamex under way |
| COP-EOG | Oil & Gas Exploration & Production | 40 | 0.87 | 7.0% | 0.218 | 55 | COP acquired Marathon Oil (2024) |

- **MA-V**: the two large card payment networks: same business model, same customers, revenues tied to consumer spending.
- **CSX-UNP**: two US freight railroads: same freight volumes, same regulation, same costs (fuel, labour).
- **PYPL-XYZ**: digital payments for consumers and merchants (XYZ is Block): common sector, but very wide spread and low correlation.
- **HD-LOW**: the two large US home-improvement and DIY retailers: same customers, same seasonality, same sensitivity to the housing market.
- **DHR-TMO**: life sciences tools and services: same customers (labs, pharma, biotech) and same spending cycles.
- **CRM-WDAY**: subscription enterprise software (CRM and human capital management): same customers and same sensitivity to IT spending.
- **JPM-WFC**: two large US universal banks: same rate, credit and regulatory factors.
- **NXPI-TXN**: analog and power semiconductors: same end markets (autos, industry) and same cycle.
- **ELV-UNH**: US health insurers (managed care): same premium revenues, same regulation and same cost of care.
- **CL-PG**: household and personal care consumer goods: same sales channels (mass retail) and same raw-material costs.
- **GS-MS**: the two large US investment banks: same revenues (trading, advisory, wealth management).
- **ISRG-SYK**: medical technology for surgery: same customers (hospitals) and same capital spending; different businesses (robots versus implants).
- **KO-PEP**: Coca-Cola and Pepsi: same soft-drink market, same customers and sales channels.
- **BAC-C**: two large US universal banks: same rate and credit factors.
- **COP-EOG**: oil and gas production (exploration and production): revenues tied to the same commodity prices.

### What emerges
- 5 pairs out of 15 were already in version 1 (MA-V, CSX-UNP, PYPL-XYZ, HD-LOW, KO-PEP); the other 10 are new.
- The statistical link on daily data is weak: only 1 pair has a cointegration p-value below 0.05 (4 below 0.10). The list is a list of candidates, not a proof: the proof comes from the next steps.
- Some pairs have an undefined half-life (the spread does not go back to the mean in the period) or a very different distance between the two halves of the period: see the table of part 6. They were not discarded, by rule.
- Pairs examined: 249 (multiple testing: some are close by chance).

## 10. Saving

The list of version 2 (15 pairs) goes into a **new** file (`selected_pairs_2.csv`) so as not to overwrite the one of version 1.

In [10]:
out = final.merge(diag, on="pair")
out = out[["pair", "A", "B", "sub_industry", "industry_group", "rank_ssd", "ssd", "corr", "dv_min_m", "coint_p", "beta", "half_life_days", "crossings_per_year"]]
out.to_csv(SELECTED, index=False)
out.round(3)

,pair,A,B,sub_industry,industry_group,rank_ssd,ssd,corr,dv_min_m,coint_p,beta,half_life_days,crossings_per_year
0,MA-V,MA UN Equity,V UN Equity,Transaction & Payment Processing Services,Financial Services,1,3.854,0.880,319.430,0.081,0.985,32.121,14.457
1,CSX-UNP,CSX UW Equity,UNP UN Equity,Rail Transportation,Transportation,3,6.610,0.717,137.024,0.505,0.950,56.479,8.433
2,PYPL-XYZ,PYPL UW Equity,XYZ UN Equity,Transaction & Payment Processing Services,Financial Services,4,8.737,0.612,119.005,0.557,0.985,62.121,13.453
3,HD-LOW,HD UN Equity,LOW UN Equity,Home Improvement Retail,Consumer Discretionary Distribution & Retail,5,10.391,0.868,169.109,0.144,0.975,52.404,7.630
4,DHR-TMO,DHR UN Equity,TMO UN Equity,Life Sciences Tools & Services,"Pharmaceuticals, Biotechnology & Life Sciences",6,15.916,0.808,165.605,0.286,0.907,61.593,10.642
5,CRM-WDAY,CRM UN Equity,WDAY UW Equity,Application Software,Software & Services,7,18.261,0.650,154.315,0.401,0.822,67.857,10.241
6,JPM-WFC,JPM UN Equity,WFC UN Equity,Diversified Banks,Banks,8,20.093,0.755,261.051,0.172,1.167,109.697,6.425
7,NXPI-TXN,NXPI UW Equity,TXN UW Equity,Semiconductors,Semiconductors & Semiconductor Equipment,9,21.583,0.756,186.356,0.790,0.885,66.262,10.441
8,ELV-UNH,ELV UN Equity,UNH UN Equity,Managed Health Care,Health Care Equipment & Services,10,22.842,0.628,137.375,0.114,0.645,60.335,10.843
9,CL-PG,CL UN Equity,PG UN Equity,Household Products,Household & Personal Products,17,33.171,0.720,109.750,0.065,1.139,62.264,8.835
